In [66]:
%load_ext autoreload
%autoreload 2

import numpy as np
import polars as pl
import torch
import torch_geometric as pyg
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variance_filtering, class_variational_selection
from bipartite_gnn.preprocessing import mrmr_selection
from baseline_evals.knn_eval import knn_eval
from baseline_evals.svm_eval import svm_eval
from baseline_evals.xgboost_eval import xgboost_eval
from baseline_evals.mlp_eval import mlp_eval

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [70]:
mrna = pl.read_csv('MDS_data_preprocessed/mrna.csv')
mirna = pl.read_csv('MDS_data_preprocessed/mirna.csv')
circrna = pl.read_csv('MDS_data_preprocessed/circrna.csv')
te_counts = pl.read_csv('MDS_data_preprocessed/te_counts.csv')
pirna = pl.read_csv('MDS_data_preprocessed/pirna.csv')

annot = pl.read_csv('MDS_data_preprocessed/annotations.csv')

In [68]:
sample_names = annot['SAMPLE_NAME'].to_list()
sample_names = [name.split('_')[0] for name in sample_names]

assert sample_names == mrna.columns[1:]  == mirna.columns[1:] == circrna.columns[1:] == te_counts.columns[1:]

In [25]:
# prepare pirnas too
# pirna = pl.read_excel("MDS_data/piRNA_counts.xlsx")

# pirna_samples = pirna.columns[1:]

# pirna_samples = [s.split("_")[0] for s in pirna_samples]

# common_samples_w_pirna = list(set(pirna_samples).intersection(set(sample_names)))

# annot_pi = annot.filter(pl.col("sample_ids").is_in(common_samples_w_pirna))

# pirna.columns = ['piRNA'] + pirna_samples
# pirna = pirna.select(['piRNA'] + common_samples_w_pirna)
# pirna_X = pirna.to_numpy()[:, 1:]
# pirna_zeros_mask = pirna_X.sum(axis=1) < 300
# pirna_zeros_idx = np.where(pirna_zeros_mask == False)[0]
# pirna_names = pirna['piRNA'].to_list()
# pirna_names = [name.split('/')[0] for name in pirna_names]
# pirna = pirna.with_columns(piRNA=pl.Series(pirna_names))
# pirna[pirna_zeros_idx].write_csv("MDS_data_preprocessed/pirna.csv")

piRNA,V1565_S17-UMIs,N58_S10-UMIs,V1874_S15-UMIs,V777_S8-UMIs,N80_S20-UMIs,V1788_S3-UMIs,N65_S19-UMIs,V2368_S6-UMIs,N81_S21-UMIs,N59_S18-UMIs,V2286_S2-UMIs,V406_S7-UMIs,V100_S12-UMIs,N82_S1-UMIs,V2133_S13-UMIs,V574_S16-UMIs,V2115_S9-UMIs,V1921_S11-UMIs,V714_S4-UMIs,V637_S14-UMIs,V1742_S5-UMIs,V1744_S2-UMIs,V2248_S3-UMIs,V1428_S4-UMIs,V18_S5-UMIs,V1857_S2-UMIs,V839_S13-UMIs,V912_S21-UMIs,V1048_S9-UMIs,V911_S10-UMIs,V940_S13-UMIs,V681_S21-UMIs,V708_S24-UMIs,N60_S12-UMIs,N70_S11-UMIs,V148_S12-UMIs,…,V1441_S29-UMIs,V1699_S1-UMIs,V1297_S1-UMIs,V1321_S15-UMIs,V1505_S30-UMIs,V1249_S18-UMIs,V1456_S8-UMIs,V1426_S26-UMIs,V1394_S27-UMIs,V1592_S15-UMIs,V1528_S14-UMIs,V1591_S22-UMIs,V833_S6-UMIs,V1708_S27-UMIs,V1800_S16-UMIs,V1776_S23-UMIs,V1823_S11-UMIs,V1775_S15-UMIs,V1834_S3-UMIs,V2378_S19-UMIs,V2414_S10-UMIs,V1860_S4-UMIs,V1884_S28-UMIs,V1920_S7-UMIs,V2322_S18-UMIs,V2311_S18-UMIs,V2291_S17-UMIs,V1957_S6-UMIs,V2092_S17-UMIs,V2284_S16-UMIs,V2278_S20-UMIs,V2110_S7-UMIs,V2179_S9-UMIs,V2147_S8-UMIs,V2224_S19-UMIs,V2089_S1-UMIs,V788_S2-UMIs
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""hsa_piR_006779…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""hsa_piR_007653…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""hsa_piR_009540…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""hsa_piR_000302…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""hsa_piR_000390…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""hsa_piR_020814…",315,274,230,160,333,154,116,192,213,244,83,147,332,275,94,50,239,168,515,14,122,162,188,111,309,573,18,18,50,59,45,98,70,133,203,138,…,177,108,225,170,193,210,79,42,418,327,144,70,209,246,319,128,88,160,214,139,183,87,94,127,212,78,136,187,67,154,203,222,139,273,161,175,43
"""hsa_piR_020815…",316,458,493,422,279,266,140,417,243,395,220,74,361,450,77,120,715,170,210,12,289,688,336,203,604,137,104,193,270,276,265,202,410,132,225,291,…,340,201,157,1068,389,405,177,89,1005,578,329,314,574,551,458,187,117,421,669,646,183,250,170,131,635,248,170,864,126,293,571,231,224,281,574,183,120
"""hsa_piR_020829…",2732,1064,1346,610,1598,894,848,528,1525,1060,810,565,824,1131,380,1242,833,1493,2428,146,901,1991,378,714,3829,766,226,175,544,380,632,547,312,589,472,1735,…,65,1050,383,193,1198,264,610,112,492,475,1173,1241,330,385,292,375,506,325,687,330,1212,777,665,138,1620,357,621,874,573,661,1069,1170,784,749,1192,921,391


In [69]:
y_d = annot['1 disease'].to_numpy()
y_r = annot['2 risk'].to_numpy()
y_m = annot['3 mutations (SF3B1only_wt)'].to_numpy()

np.unique(y_d, return_counts=True), np.unique(y_r, return_counts=True), np.unique(y_m, return_counts=True)

((array([1, 2]), array([13, 61])),
 (array([0, 1, 2]), array([21, 29, 24])),
 (array([0, 1, 2]), array([48, 14, 12])))

In [14]:
mrna_names = mrna.columns[1:]
mirna_names = mirna.columns[1:]
circrna_names = circrna.columns[1:]
te_names = te_counts.columns[1:]

y_d = annot['1 disease'].to_numpy()
y_r = annot['2 risk'].to_numpy()
y_m = annot['3 mutations (SF3B1only_wt)'].to_numpy()

y = y_d

annotated_samples_mask = y != 0

In [15]:
mrna_X = mrna.to_numpy()[:, 1:].T
mirna_X = mirna.to_numpy()[:, 1:].T
circrna_X = circrna.to_numpy()[:, 1:].T
te_X = te_counts.to_numpy()[:, 1:].T

mrna_X = mrna_X[annotated_samples_mask]
mirna_X = mirna_X[annotated_samples_mask]
circrna_X = circrna_X[annotated_samples_mask]
te_X = te_X[annotated_samples_mask]

y = y[annotated_samples_mask]

In [16]:
np.unique(y, return_counts=True)

(array([1, 2]), array([13, 61]))

In [71]:
X = np.concatenate([mrna_X, mirna_X, circrna_X, te_X], axis=1)

In [72]:
X.shape

(74, 22508)